In [1]:
from pathlib import Path                    # Obsługa projektu
import pandas as pd
import numpy as np
import statsmodels.tsa.stattools as ts
import statsmodels.api as sm
from statsmodels.tsa.filters.hp_filter import hpfilter
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
# ── Konfiguracja 2: Wczytanie danych z 01_etl ──────────────────────

# Konfiguracja ścieżek (zgodna z pierwszym notebookiem)
PROJECT_ROOT = Path.cwd().parent  # ponieważ notebook jest w notebooks/
DATA_DIR = PROJECT_ROOT / "processed"  # uwaga: bez "data" w środku

OUT_DIR = PROJECT_ROOT / "outputs" / "html"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Odczyt danych
df = pd.read_parquet(DATA_DIR / "df.parquet")
df_rok = pd.read_parquet(DATA_DIR / "df_rok.parquet")

In [8]:
# ── SEKCJA 5: HP FILTER — dekompozycja trendu i cyklu ────────────────────────


# ── KOLORY ────────────────────────────────────────────────────────────────────
CZERWONY   = '#C0392B'
NIEBIESKI  = '#1B3A6B'
ZLOTY      = '#C9982A'
ZIELONY    = '#1E8449'
TURKUSOWY  = '#1A7A6E'
MGLA       = '#8395A7'
SZARY_GRID = '#E4EAF2'
BG         = '#F7F9FC'
BIALY      = '#FFFFFF'

# ══════════════════════════════════════════════════════════════════════════════
# LOGIKA HP FILTER
#
# Pytanie badawcze 4: Czy wzrost VAT/PKB po reformach jest trwały czy cykliczny?
#
# Metoda Hodricka-Prescotta (1997):
#   y_t = τ_t + c_t
#   gdzie:
#     τ_t = komponent trendu (długookresowy)
#     c_t = komponent cykliczny (krótkookresowy)
#
# Parametr lambda:
#   λ = 1600 — standard dla danych kwartalnych (Hodrick-Prescott 1997)
#   Wyższe λ → gładszy trend → więcej zmienności w cyklu
#   Niższe λ  → trend bliższy danym → mniej zmienności w cyklu
#
# Interpretacja dla projektu:
#   Jeśli τ_t (trend) rośnie po 2016Q3 → wzrost TRWAŁY
#   Jeśli c_t (cykl) rośnie po 2016Q3 → wzrost CYKLICZNY (przejściowy)
# ══════════════════════════════════════════════════════════════════════════════

LAMBDA = 1600   # standard dla danych kwartalnych

# ── Dane ──────────────────────────────────────────────────────────────────────
y = df['vat_pkb_pct'].copy()
lata = df.index.tolist()

# ── Dekompozycja HP ───────────────────────────────────────────────────────────
cykl, trend = hpfilter(y, lamb=LAMBDA)

df['hp_trend'] = trend.values
df['hp_cykl']  = cykl.values

# ── Analiza przed/po JPK ──────────────────────────────────────────────────────
przed_mask = df.index <= '2016Q2'
po_mask    = df.index >= '2016Q3'

trend_przed = df.loc[przed_mask, 'hp_trend']
trend_po    = df.loc[po_mask,    'hp_trend']
cykl_przed  = df.loc[przed_mask, 'hp_cykl']
cykl_po     = df.loc[po_mask,    'hp_cykl']

# Nachylenie trendu przed i po JPK — czy trend przyspieszył?
t_przed = np.arange(len(trend_przed))
t_po    = np.arange(len(trend_po))

a_przed, b_przed = np.polyfit(t_przed, trend_przed, deg=1)
a_po,    b_po    = np.polyfit(t_po,    trend_po,    deg=1)

print('=' * 65)
print('  HP FILTER — DEKOMPOZYCJA TREND/CYKL')
print(f'  Lambda = {LAMBDA} (standard kwartalny)')
print('=' * 65)
print()
print('  KOMPONENT TRENDU (τ):')
print(f'  Śr. trend przed 2016Q3: {trend_przed.mean():.4f}%')
print(f'  Śr. trend po 2016Q3:    {trend_po.mean():.4f}%')
print(f'  Δ trend:                {trend_po.mean()-trend_przed.mean():+.4f} pp')
print()
print(f'  Nachylenie trendu przed: {a_przed*4:+.4f} pp/rok')
print(f'  Nachylenie trendu po:    {a_po*4:+.4f} pp/rok')
print(f'  Zmiana nachylenia:       {(a_po-a_przed)*4:+.4f} pp/rok')
print()
print('  KOMPONENT CYKLICZNY (c):')
print(f'  Śr. cykl przed 2016Q3: {cykl_przed.mean():.4f}%')
print(f'  Śr. cykl po 2016Q3:    {cykl_po.mean():.4f}%')
print(f'  Std cykl przed:         {cykl_przed.std():.4f}%')
print(f'  Std cykl po:            {cykl_po.std():.4f}%')
print()

# Interpretacja
delta_trend = trend_po.mean() - trend_przed.mean()
if delta_trend > 0.2:
    print(f'  ✅ WZROST TRWAŁY — trend wyższy po reformach o {delta_trend:.3f} pp')
    print(f'     Reformy podniosły długookresowy poziom efektywności VAT')
elif delta_trend > 0:
    print(f'  ⚠️  WZROST CZĘŚCIOWO TRWAŁY — trend wyższy o {delta_trend:.3f} pp')
else:
    print(f'  ❌ WZROST CYKLICZNY — trend nie zmienił się po reformach')
print('=' * 65)

# ── Robustness check — różne lambda ──────────────────────────────────────────
print('\n=== ROBUSTNESS CHECK — różne λ ===')
print(f'  {"Lambda":>8} {"Trend przed":>12} {"Trend po":>10} {"Δ trend":>10}')
print('  ' + '-' * 44)

for lam in [400, 1600, 6400, 25600]:
    c_tmp, t_tmp = hpfilter(y, lamb=lam)
    trend_pr = t_tmp[przed_mask].mean()
    trend_po_sr = t_tmp[po_mask].mean()

    print(
        f'  {lam:>8} '
        f'{trend_pr:>12.4f} '
        f'{trend_po_sr:>10.4f} '
        f'{trend_po_sr-trend_pr:>+10.4f}'
    )


# ══════════════════════════════════════════════════════════════════════════════
# WIZUALIZACJA — 4 panele
# ══════════════════════════════════════════════════════════════════════════════

REFORMY = [
    ('2016Q3', NIEBIESKI, 'JPK'),
    ('2018Q3', ZLOTY,     'Split'),
    ('2019Q3', ZIELONY,   'Biała lista'),
    ('2020Q2', MGLA,      'COVID'),
]

viz = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        '① vat_pkb_pct — dane i trend HP',
        '② Komponent cykliczny HP',
        '③ Trend HP przed/po 2016Q3',
        '④ Robustness — trend dla różnych λ',
    ),
    vertical_spacing=0.20,
    horizontal_spacing=0.10,
)

# ── ZBIERZ WSZYSTKIE SHAPES (rozwiązanie nr 1 z kodu 1) ─────────────────────
shapes = []

# Definicja osi dla każdego subplotu
subplot_axes = [
    ('x',  'y'),   # row 1, col 1
    ('x2', 'y2'),  # row 1, col 2
    ('x3', 'y3'),  # row 2, col 1
    ('x4', 'y4'),  # row 2, col 2
]

# Dla każdego subplotu dodaj linie reform
for xref, yref in subplot_axes:
    for rok, kolor, _ in REFORMY:
        shapes.append(dict(
            type='line',
            xref=xref, 
            yref=f'{yref} domain',  # KLUCZOWE: domain zamiast paper
            x0=rok, 
            x1=rok, 
            y0=0, 
            y1=1,
            line=dict(color=kolor, width=1.2, dash='dash'),
            opacity=0.65, 
            layer='below',
        ))

# ── Panel 1: Dane + trend ─────────────────────────────────────────────────────
viz.add_trace(go.Scatter(
    x=lata, y=y.tolist(),
    mode='lines',
    name='vat_pkb_pct',
    line=dict(color=MGLA, width=1.5),
    opacity=0.7,
    showlegend=True,
    hovertemplate='<b>%{x}</b><br>VAT/PKB: %{y:.3f}%<extra></extra>',
), row=1, col=1)

viz.add_trace(go.Scatter(
    x=lata, y=df['hp_trend'].tolist(),
    mode='lines',
    name=f'Trend HP (λ={LAMBDA})',
    line=dict(color=CZERWONY, width=2.5),
    showlegend=True,
    hovertemplate='<b>%{x}</b><br>Trend: %{y:.3f}%<extra></extra>',
), row=1, col=1)

# ── Panel 2: Cykl ────────────────────────────────────────────────────────────
kolory_cykl = [CZERWONY if v > 0 else NIEBIESKI for v in cykl.values]

viz.add_trace(go.Bar(
    x=lata, y=cykl.tolist(),
    marker_color=kolory_cykl,
    marker_line_color='white',
    marker_line_width=0.3,
    opacity=0.8,
    name='Cykl HP',
    showlegend=False,
    hovertemplate='<b>%{x}</b><br>Cykl: %{y:.3f} pp<extra></extra>',
), row=1, col=2)

viz.add_hline(y=0, row=1, col=2,
              line=dict(color='black', width=0.8))

# ── Panel 3: Trend przed/po z nachyleniem ────────────────────────────────────

lata_przed_list = df.index[przed_mask].tolist()
lata_po_list    = df.index[po_mask].tolist()

# ── FIX: jeden spójny trend (eliminuje bug Plotly z legendą)
trend_full = np.full(len(df), np.nan)
trend_full[przed_mask] = trend_przed.values
trend_full[po_mask]    = trend_po.values

viz.add_trace(go.Scatter(
    x=lata,
    y=trend_full,
    mode='lines',
    name='Trend HP',
    line=dict(color=CZERWONY, width=2.5),
    showlegend=True,
    hovertemplate='<b>%{x}</b><br>Trend: %{y:.3f}%<extra></extra>',
), row=2, col=1)

# ── Linia trendu liniowego przed
y_lin_przed = np.polyval([a_przed, b_przed], t_przed)
viz.add_trace(go.Scatter(
    x=lata_przed_list,
    y=y_lin_przed.tolist(),
    mode='lines',
    name=f'Lin. trend przed ({a_przed*4:+.4f} pp/rok)',
    line=dict(color=MGLA, width=1.2, dash='dot'),
    showlegend=True,
    hoverinfo='skip',
), row=2, col=1)

# ── Linia trendu liniowego po
y_lin_po = np.polyval([a_po, b_po], t_po)
viz.add_trace(go.Scatter(
    x=lata_po_list,
    y=y_lin_po.tolist(),
    mode='lines',
    name=f'Lin. trend po ({a_po*4:+.4f} pp/rok)',
    line=dict(color=CZERWONY, width=1.2, dash='dot'),
    showlegend=True,
    hoverinfo='skip',
), row=2, col=1)

# ── Panel 4: Robustness ───────────────────────────────────────────────────────
lambdy  = [400, 1600, 6400, 25600]
kolory_lam = [TURKUSOWY, CZERWONY, ZLOTY, NIEBIESKI]

for lam, kolor in zip(lambdy, kolory_lam):
    _, t_tmp = hpfilter(y, lamb=lam)
    viz.add_trace(go.Scatter(
        x=lata, y=t_tmp.tolist(),
        mode='lines',
        name=f'λ={lam}',
        line=dict(color=kolor, width=1.8),
        showlegend=True,
        hovertemplate=f'<b>%{{x}}</b><br>λ={lam}: %{{y:.3f}}%<extra></extra>',
    ), row=2, col=2)

# ── DODAJ ANNOTACJE DLA REFORM (opcjonalnie) ─────────────────────────────────
# Dodajemy etykiety tylko dla pierwszego subplotu (lub wybranych)
for idx, (rok, kolor, label) in enumerate(REFORMY):
    # Dla panelu 2 (cykl)
    viz.add_annotation(
        x=rok,
        xref='x2',
        yref='y2 domain',
        y=0.97 - idx * 0.03,  # Rozmieść etykiety w pionie
        text=label,
        showarrow=False,
        font=dict(size=8, color=kolor),
        xanchor='left',
    )
    
    # Dla panelu 4 (robustness)
    viz.add_annotation(
        x=rok,
        xref='x4',
        yref='y4 domain',
        y=0.97 - idx * 0.03,
        text=label,
        showarrow=False,
        font=dict(size=8, color=kolor),
        xanchor='left',
    )

# Dla panelu 3 (trend) - tylko JPK
viz.add_annotation(
    x='2016Q3',
    xref='x3',
    yref='y3 domain',
    y=0.97,
    text='JPK 2016Q3',
    showarrow=False,
    font=dict(size=9, color=NIEBIESKI),
    xanchor='left',
)

# ── LAYOUT ───────────────────────────────────────────────────────────────────
viz.update_layout(
    title=dict(
        text=(
            '<b>HP Filter · Dekompozycja trendu i cyklu — vat_pkb_pct</b><br>'
            '<span style="font-size:11px;color:#8395A7">'
            f'λ={LAMBDA} (standard kwartalny) · '
            f'Δ trend po reformach: {delta_trend:+.3f} pp · '
            f'Nachylenie trendu: {a_przed*4:+.4f}→{a_po*4:+.4f} pp/rok<br>'
            'Opracowanie: Mateusz Durski · 2026'
            '</span>'
        ),
        x=0.01, xanchor='left',
        font=dict(size=15),
    ),
    height=1000,
    paper_bgcolor=BG,
    hovermode='x unified',
    template='simple_white',
    shapes=shapes,  # <--- DODAJ WSZYSTKIE SHAPES
    legend=dict(
        orientation='h',
        y=-0.15, x=0.5, xanchor='center',
        font=dict(size=9),
        bgcolor='rgba(247,249,252,0.9)',
        bordercolor=SZARY_GRID, borderwidth=1,
    ),
    margin=dict(t=120, b=180, l=80, r=60),
)

# ── OSIE ─────────────────────────────────────────────────────────────────────
for row, col, ylabel in [
    (1, 1, 'vat_pkb_pct (%)'),
    (1, 2, 'Odchylenie cykliczne (pp)'),
    (2, 1, 'Trend HP (%)'),
    (2, 2, 'Trend HP (%)'),
]:
    viz.update_yaxes(title_text=ylabel, showgrid=True,
                     gridcolor=SZARY_GRID, row=row, col=col)
    viz.update_xaxes(title_text='Kwartał', tickangle=-45,
                     showgrid=False, row=row, col=col)

# ── EXPORT ───────────────────────────────────────────────────────────────────
file_path = OUT_DIR / "hp_filter.html"
viz.write_html(
    file_path,
    include_plotlyjs='cdn',
)
print('\n✅ Zapisano: hp_filter.html')
viz.show()

  HP FILTER — DEKOMPOZYCJA TREND/CYKL
  Lambda = 1600 (standard kwartalny)

  KOMPONENT TRENDU (τ):
  Śr. trend przed 2016Q3: 7.3034%
  Śr. trend po 2016Q3:    7.7260%
  Δ trend:                +0.4226 pp

  Nachylenie trendu przed: +0.0193 pp/rok
  Nachylenie trendu po:    +0.0444 pp/rok
  Zmiana nachylenia:       +0.0251 pp/rok

  KOMPONENT CYKLICZNY (c):
  Śr. cykl przed 2016Q3: -0.0244%
  Śr. cykl po 2016Q3:    0.0449%
  Std cykl przed:         0.3940%
  Std cykl po:            0.6451%

  ✅ WZROST TRWAŁY — trend wyższy po reformach o 0.423 pp
     Reformy podniosły długookresowy poziom efektywności VAT

=== ROBUSTNESS CHECK — różne λ ===
    Lambda  Trend przed   Trend po    Δ trend
  --------------------------------------------
       400       7.2925     7.7459    +0.4534
      1600       7.3034     7.7260    +0.4226
      6400       7.3154     7.7037    +0.3883
     25600       7.3214     7.6927    +0.3713

✅ Zapisano: hp_filter.html
